# Δημιουργία Dataset & Φόρτωση στη MongoDB

Αυτό το notebook καλύπτει δύο βασικά βήματα της προεργασίας μας:

1. **Συγχώνευση (merge)** των ξεχωριστών αρχείων επιταχυνσιόμετρου (_Acc_) και γυροσκοπίου (_Gyr_) σε ένα ενιαίο αρχείο 6 αξόνων (_AccGyr_).
2. **Φόρτωση** των τελικών CSV στη MongoDB ώστε να είναι έτοιμα για επεξεργασία από τα υπόλοιπα notebooks.

> **Προσοχή:** Βεβαιωθείτε ότι ο MongoDB server τρέχει στο παρασκήνιο πριν εκτελέσετε οποιοδήποτε κελί.

In [1]:
# import library for various processes with the OS
import os

## Load configuration

In [2]:
# import library for yaml handling
import yaml

Φορτώνουμε το αρχείο `config.yml` που περιέχει όλες τις παραμέτρους του project (διαδρομές, ρυθμίσεις MongoDB, παράμετροι φίλτρων κ.λπ.). Έτσι αποφεύγουμε να έχουμε hardcoded τιμές μέσα στον κώδικα.

In [3]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

Εκτυπώνουμε τη διαδρομή του φακέλου δεδομένων για να επιβεβαιώσουμε ότι το config φορτώθηκε σωστά.

In [4]:
print(config["data_path"])

data


## Merge Αρχείων Επιταχυνσιόμετρου & Γυροσκοπίου

Ο αισθητήρας MetaMotionR αποθηκεύει τα δεδομένα του επιταχυνσιόμετρου και του γυροσκοπίου σε **ξεχωριστά αρχεία CSV** (π.χ. `scroll-down_Acc_....csv` και `scroll-down_Gyr_....csv`). Για να δουλέψουμε με ένα ενιαίο 6-αξονικό σήμα, πρέπει να τα συγχωνεύσουμε.

Η διαδικασία είναι:
- Βρίσκουμε κάθε αρχείο `_Acc_` μέσα στον φάκελο δεδομένων.
- Βρίσκουμε το αντίστοιχο `_Gyr_` αρχείο (ίδιο όνομα, διαφορετικό prefix).
- Κάνουμε `df_rebase` και στα δύο για να ευθυγραμμίσουμε τις στήλες σύμφωνα με το config.
- Κάνουμε `merge` στη στήλη `epoc (ms)` ώστε τα δείγματα να ταιριάξουν χρονικά.
- Κρατάμε μόνο τις 6 τελικές στήλες: `acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z`.
- Αποθηκεύουμε το νέο αρχείο με πρόθεμα `_AccGyr_`.

Αν το merged αρχείο υπάρχει ήδη, το παραλείπουμε για να μην κάνουμε διπλή δουλειά.

In [5]:

import pandas as pd
from utils import df_rebase

for root, dirs, files in os.walk(config["data_path"]):
    acc_files = [f for f in files if "_Acc_" in f]
    
    for f_acc in acc_files:
        path_acc = os.path.join(root, f_acc)
        path_gyr = path_acc.replace("_Acc_", "_Gyr_")

        new_name = f_acc.replace("_Acc_", "_AccGyr_")
        output_path = os.path.join(root, new_name)

        if os.path.exists(output_path):
            print(f"Skipping {new_name}, already merged.")
            continue
        
        if os.path.exists(path_gyr):
            df_acc = pd.read_csv(path_acc)
            df_gyr = pd.read_csv(path_gyr)
            
            df_acc = df_rebase(df_acc, config["order"], config["rename"])
            df_gyr = df_rebase(df_gyr, config["order_gyr"], config["rename_gyr"])
            
            df_merged = pd.merge(df_acc, df_gyr, on="epoc (ms)")
            
            final_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
            df_final = df_merged[final_cols]
            
            new_name = f_acc.replace("_Acc_", "_AccGyr_")
            df_final.to_csv(os.path.join(root, new_name), index=False)
            print("Processing data...")

Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)']
Processed columns: ['epoc (ms)', 'acc_x', 'acc_y', 'acc_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['epoc (ms)', 'gyr_x', 'gyr_y', 'gyr_z']
Processing data...
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)']
Processed columns: ['epoc (ms)', 'acc_x', 'acc_y', 'acc_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['epoc (ms)', 'gyr_x', 'gyr_y', 'gyr_z']
Processing data...
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)']
Processed columns: ['epoc (ms)', 'acc_x', 'acc_y', 'acc_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (deg/s)', 'y-axis 

## Καθαρισμός Αρχικών Αρχείων

Αφού ολοκληρωθεί το merge, διαγράφουμε τα ξεχωριστά αρχεία `_Acc_` και `_Gyr_` ώστε ο φάκελος να περιέχει μόνο τα τελικά `_AccGyr_` CSV. Αυτό μειώνει τον χώρο αποθήκευσης και αποφεύγει σύγχυση στα επόμενα βήματα.

In [11]:

for root, dirs, files in os.walk(config["data_path"]):
    acc_files = [f for f in files if "_Acc_" in f]
    
    for f_acc in acc_files:
        path_acc = os.path.join(root, f_acc)
        path_gyr = path_acc.replace("_Acc_", "_Gyr_")
        os.remove(path_acc)
        os.remove(path_gyr)
        

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file.

```
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "NAME YOUR COLLECTION"
```

Εισάγουμε τη βιβλιοθήκη `pymongo` για τη σύνδεση με τη MongoDB, και το `datetime` για να καταγράφουμε την ακριβή ώρα κάθε εγγραφής.

In [6]:
# import library for hanlding the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a MongoClient object, then specify a connection URL with the correct ip address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

Ανοίγουμε σύνδεση με τον local MongoDB server στη διεύθυνση που ορίζει το config (`mongodb://localhost:27017/`). Αν ο server δεν τρέχει, εδώ θα εμφανιστεί σφάλμα.

In [7]:
client = pymongo.MongoClient(config["client"])

Επιλέγουμε (ή δημιουργούμε αν δεν υπάρχει) τη βάση δεδομένων με το όνομα που ορίζεται στο `config.yml`.

In [8]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use the database object and specify the name of the collection you want to create.

MongoDB will create the collection if it does not exist.

Ομοίως, επιλέγουμε την collection μέσα στη βάση. Η MongoDB δημιουργεί αυτόματα collection και βάση μόλις εισαχθεί το πρώτο έγγραφο — μέχρι τότε δεν φαίνονται πουθενά.

In [9]:
col = db[config["col"]]

Initially, no collection will be shown in MongoDB before you enter the first document!

## Create the data collection

Uploading the gathered data to MongoDB collection. The data directory structure should be as follows:

```
.
└── data/
    ├──
    ├── scroll-up-thumb/
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_61964f51d11ec26c3bbde60b.csv
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_7982f51d11ec26c3be60b875.csv
    │   └── ..
    ├── scroll-up-down/
    │   ├── scroll-up-down_0_50_AccGyr_1_1_01_12432f51d11ec26c3b944jd0.csv
    │   ├── scroll-up-down_0_50_AccGyr_1_1_02_47563f51d11ec26c3bb4210k.csv
    │   └── .
    └── class ...
```

In [10]:
# import library for handling the csv data and transformations
import pandas as pd
import json

Get data path:

Χτίζουμε την απόλυτη διαδρομή στον φάκελο `data/` που βρίσκεται δίπλα στο notebook.

In [11]:
data_path = os.path.join(os.getcwd(), "data")
print(data_path)

c:\IoT-Course-AIoT-project\data


List all files in a path:

Φτιάχνουμε μια λίστα με τους υποφακέλους του `data/` — κάθε υποφάκελος αντιστοιχεί σε μία κλάση κίνησης.

In [12]:
classes_folders_list = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]
print(classes_folders_list)

['scroll-down-index', 'scroll-up-index', 'swipe-left-index', 'swipe-right-index', 'texting-single']


Ελέγχουμε ποια αρχεία βρίσκονται στον πρώτο φάκελο κλάσης, για να επιβεβαιώσουμε ότι τα merged CSV είναι εκεί.

In [13]:
# print files in folder
folder_path = os.path.join(data_path, classes_folders_list[0])
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(files_in_folder)

['scroll-down-index_0_100_AccGyr_1_1_01_seg01.csv', 'scroll-down-index_0_100_AccGyr_1_1_01_seg02.csv', 'scroll-down-index_0_100_AccGyr_1_1_01_seg03.csv', 'scroll-down-index_0_100_AccGyr_1_1_01_seg04.csv', 'scroll-down-index_0_100_AccGyr_1_1_01_seg05.csv', 'scroll-down-index_0_100_AccGyr_1_1_02_seg01.csv', 'scroll-down-index_0_100_AccGyr_1_1_02_seg02.csv', 'scroll-down-index_0_100_AccGyr_1_1_02_seg03.csv', 'scroll-down-index_0_100_AccGyr_1_1_02_seg04.csv', 'scroll-down-index_0_100_AccGyr_1_1_02_seg05.csv']


Each document in the MongoDB database should have the following schema:

```json
{
  "_id": ObjectId("6984b3fa87abe7f4dff571aa"),
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"]
  },
  "gesture_id": "The label of the instance",
  "hand": 0,
  "sr": 50,
  "sensor": "AccGyr",
  "primary": 1,
  "spontaneous": 1,
  "user": "01",
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function"
}
```

Accordingly, if you are using gyroscope or both accelerometer and gyroscope, the following order and naming of the sensor keys should be defined:

* for gyroscope: `gyr_x`, `gyr_y`, `gyr_z` for the three axes
* for accelerometer and gyroscope: `acc_x`, `acc_y`, `acc_z`, `gyr_x`, `gyr_y`, `gyr_z` for the six axes

**Note: Be careful, the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.**

Εισάγουμε ξανά τη συνάρτηση `df_rebase` που χρειαζόμαστε για τη σωστή ευθυγράμμιση των στηλών κατά τη φόρτωση.

In [14]:
from utils import df_rebase

⚠️ **Προσοχή:** Το παρακάτω κελί **διαγράφει** εντελώς την collection από τη MongoDB. Το τρέχουμε μόνο όταν θέλουμε να ξεκινήσουμε από την αρχή (π.χ. μετά από επανεγγραφή δεδομένων). Σε κανονική χρήση το παραλείπουμε.

In [15]:
col.drop()
print("Collection dropped successfully. Ready for fresh upload!")

Collection dropped successfully. Ready for fresh upload!


## Provide the code to upload the data to MongoDB

## Φόρτωση Δεδομένων στη MongoDB

Εδώ γίνεται το κύριο upload. Για κάθε αρχείο `_AccGyr_` CSV:

- Αναλύουμε το όνομα αρχείου (π.χ. `scroll-down-index_0_100_AccGyr_1_0_01_<id>.csv`) για να εξάγουμε τα metadata (κίνηση, χέρι, sampling rate, sensor, κ.λπ.).
- Φορτώνουμε το CSV σε DataFrame και το μετατρέπουμε σε λεξικό (`orient='list'`) ώστε κάθε στήλη να γίνει λίστα τιμών.
- Δημιουργούμε ένα MongoDB document με το προκαθορισμένο schema (`data`, `gesture_id`, `hand`, `sr`, `sensor`, `primary`, `spontaneous`, `user`, `datetime`).
- Εισάγουμε το document στην collection με `insert_one`.

Κάθε αρχείο CSV αντιστοιχεί σε **μία εγγραφή (session)** στη βάση.

In [16]:
classes_folders_list = [f for f in os.listdir(config["data_path"]) if os.path.isdir(os.path.join(config["data_path"], f))]

for folder_name in classes_folders_list:
    current_folder_path = os.path.join(config["data_path"], folder_name)
    files_in_folder = [f for f in os.listdir(current_folder_path) if os.path.isfile(os.path.join(current_folder_path, f))]
    
    for file in files_in_folder:
        if "_AccGyr_" in file and file.endswith(".csv"):
            parts = file.replace(".csv", "").split("_")

            query = {
                "gesture_id": parts[0],
                "hand": int(parts[1]),
                "user": parts[6],
                "filename": file 
            }

            df = pd.read_csv(os.path.join(current_folder_path, file))
            sensor_data = df.to_dict(orient='list')
            
            document = {
                "data": sensor_data,
                "gesture_id": parts[0],
                "hand": int(parts[1]),
                "sr": int(parts[2]),
                "sensor": parts[3],
                "primary": int(parts[4]),
                "spontaneous": int(parts[5]),
                "user": parts[6],
                "datetime": datetime.now()
            }
            
            col.insert_one(document)
            print(f"Uploaded: {file} from folder: {folder_name}")

Uploaded: scroll-down-index_0_100_AccGyr_1_1_01_seg01.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_01_seg02.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_01_seg03.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_01_seg04.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_01_seg05.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_02_seg01.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_02_seg02.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_02_seg03.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_02_seg04.csv from folder: scroll-down-index
Uploaded: scroll-down-index_0_100_AccGyr_1_1_02_seg05.csv from folder: scroll-down-index
Uploaded: scroll-up-index_0_100_AccGyr_1_1_01_seg01.csv from folder: scroll-up-index
Uploaded: scroll-up-index

Η συλλογή καθαρίστηκε επιτυχώς.
